# Development preprocessing and five-minute IBI eligibility

This notebook reads privacy-reviewed aggregate reports. The reusable pipeline preserves raw development EMA and Garmin records, then audits window eligibility under three provisional policies and both interval-boundary hypotheses. It does not calculate PSS targets or HRV, impute values, train models, or explore final-test records.

The frozen split remains 125 development / 32 final test, with five original development folds. Raw values, identifying provenance, split assignments and window-level tables stay under ignored `private/`; this source notebook must remain output-free.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
REPORT = ROOT / "reports/ibi_eligibility/6747223eb50269ba_90911a7ae670"
CONFIG = json.loads((ROOT / "config/ibi_eligibility.json").read_text())

## Optional reproducible pipeline run

Set `GARMIN_RUN_PREPROCESSING=1` and `GARMIN_REFERENCE_PROJECT` to run the development pipeline. `GARMIN_DATA_DIR` may override the local raw archive directory. Existing validated versions are hash-checked and reused; changed inputs or code create a new version. Failures omit source records. The aggregate walkthrough below is pinned to this validated checkpoint and will not silently switch versions.

In [ ]:
if os.environ.get("GARMIN_RUN_PREPROCESSING") == "1":
    env = dict(os.environ, PYTHONPATH=str(ROOT / "src"))
    command = [sys.executable, "-m", "garmin_preprocessing.audit"]
    if os.environ.get("GARMIN_DATA_DIR"):
        command += ["--data", os.environ["GARMIN_DATA_DIR"]]
    result = subprocess.run(command, cwd=ROOT, env=env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Development audit stopped; inspect the private local run log.")
    print("Development audit completed; immutable outputs verified or created.")
else:
    print("Aggregate walkthrough; no raw data loaded.")

In [ ]:
summary = json.loads((REPORT / "run_summary.json").read_text())
coverage = pd.DataFrame(json.loads((REPORT / "coverage.json").read_text()))
attrition = pd.DataFrame(json.loads((REPORT / "attrition.json").read_text()))
timing = json.loads((REPORT / "timing.json").read_text())
assert hashlib.sha256((ROOT / "config/ibi_eligibility.json").read_bytes()).hexdigest() == summary["inputs"]["policy_sha256"]
assert summary["final_test_rows_processed"] == 0
assert not summary["targets_or_hrv_calculated"]
assert summary["all_development_ema_accounted"]
display(pd.DataFrame([dict(
    development_ema=summary["ema_rows"],
    development_participants=summary["development_participants"],
    physical_sensor_rows=summary["normalization"]["counts"]["physical_sensor_rows"],
    logical_sensor_rows=summary["normalization"]["counts"]["logical_sensor_rows"],
    five_minute_window_instances=summary["window_rows"],
    ema_policy_scenario_lookback_rows=summary["ema_accounting_rows"],
)]))
display(Markdown("Ignored dataset roots: " + "; ".join(f"`{p}`" for p in summary["datasets"].values())))

## Preservation, clocks and duplicate handling

Physical Parquet records keep every original field/value, source hashes and one-based row numbers, mapping provenance, frozen fold, parsed values, clocks and flags. Sensor normalization uses chunks and bounded participant/sensor partitions. The logical view collapses only exact raw-record duplicates and points back to physical rows. Equal interval values at different timestamps remain legitimate records; timestamp ties and conflicting values remain flagged and are never averaged.

EMA UTC equals local start minus the row's local-minus-UTC offset. Garmin Unix seconds receive no additional shift. The cutoff subtracts 60 seconds only for minute-resolution starts. Clock disagreements remain flagged, never shifted to improve alignment.

For cutoff `c`, windows are `[c−60j−300, c−60j)`. For a lookback of `L` minutes, only `j=0,…,L−5` is included. Both interval-end `[t−r,t]` and interval-start `[t,t+r]` hypotheses require the timestamp and entire hypothesized interval to fit. Boundary exclusions are counted, not repaired. All normal-beat status remains unverified.

## Policies fixed before comparing retention

The numerical thresholds are research sensitivities, not validated eligibility rules. They were not chosen using PSS associations or desired sample size. The inventory's two-second / 60-second diagnostics are not these eligibility policies. See `reports/ibi_eligibility_policies.md` for official package documentation, methodological sources and exact definitions.

In [ ]:
display(pd.DataFrame(CONFIG["policies"]).set_index("name").T)
display(Markdown((ROOT / "reports/ibi_eligibility_policies.md").read_text()))

## Timestamp evidence

Consecutive timestamp differences are compared with current and preceding IBI durations, including ties, gaps and compressed-spacing bursts. Agreement can be consistent with an interval-boundary interpretation without proving acquisition semantics. No timestamps are reconstructed and disconnected recordings are never concatenated.

In [ ]:
display(pd.DataFrame.from_dict(timing["counts"], orient="index", columns=["count"]))
display(pd.DataFrame(timing["quantiles"]).T)
display(Image(filename=str(REPORT / "timing_agreement.png")))

## Coverage and attrition

All development EMA records remain in the denominators, including those without Garmin or eligible windows. Unique supported time is the union of acceptable hypothetical interval support within participant; overlapping windows and EMA histories count once. This differs from nominal eligible-window duration. Survey categories can overlap in time and must not be added. One segment in 24 hours is not complete 24-hour coverage.

In [ ]:
all_scope = coverage.loc[coverage.survey_category.eq("All")].copy()
all_scope["unique_supported_minutes"] = all_scope.unique_supported_seconds / 60
all_scope["eligible_ema_percent"] = 100 * all_scope.eligible_ema / all_scope.ema_denominator
assert all_scope.ema_denominator.eq(summary["ema_rows"]).all()
assert coverage.unique_supported_seconds.le(coverage.unique_eligible_window_seconds + 1e-4).all()
assert coverage.unique_eligible_window_seconds.le(coverage.unique_requested_seconds + 1e-4).all()
display(all_scope[["policy", "scenario", "lookback_minutes", "eligible_windows", "eligible_ema",
                   "ema_denominator", "eligible_participants", "unique_supported_minutes"]])
display(Image(filename=str(REPORT / "coverage.png")))

In [ ]:
display(coverage.loc[coverage.survey_category.ne("All"), [
    "policy", "scenario", "lookback_minutes", "survey_category", "ema_denominator",
    "eligible_windows", "eligible_ema", "eligible_participants", "unique_supported_seconds",
    "unique_requested_seconds", "maximum_window_internal_gap_seconds"]])
# Rejection reasons overlap; these counts are not sequential attrition and cannot be summed.
display(attrition.loc[attrition.lookback_minutes.eq(5) & attrition.survey_category.eq("All")])

## Verification and next implementation step

Run `PYTHONPATH=src uv run python -m garmin_preprocessing.verify` with the reference location configured. It checks original input integrity, immutable bundles, source row accounting, participant/fold separation, all-EMA accounting, aggregate reconciliation, deterministic window-kernel replay and public privacy checks. Run `uv run pytest` and `uv run ruff check src tests`; keep executed notebooks in ignored `outputs/`.

Next, implement and validate a segment-aware HRV input contract with synthetic package checks. Preserve both timing hypotheses and rejected-record boundaries until justified otherwise; never pass disconnected intervals as an uninterrupted beat sequence. Real HRV calculation is a later step. Garmin stress derivatives remain excluded from initial predictors; quality measures remain separate. No winner is selected by retention alone.